In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.kai.kaiticketing',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


In [ ]:
with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('ulasan_google_play.csv')

print("Data loaded successfully. Here are the first 5 rows:")
display(df.head())

In [ ]:
pip install transformers

In [ ]:
from transformers import pipeline

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-prdect-id")

In [ ]:
def get_sentiment(text):
    if pd.isna(text):
        return {'label': 'neutral', 'score': 0.0} # Handle NaN or empty content
    try:
        # The model returns a list of dictionaries, so we take the first one
        result = sentiment_pipeline(text)[0]
        return result
    except Exception as e:
        print(f"Error processing text: {text[:50]}... Error: {e}")
        return {'label': 'error', 'score': 0.0}

df['sentiment_analysis'] = df['content'].apply(get_sentiment)

In [ ]:
df['sentiment_label'] = df['sentiment_analysis'].apply(lambda x: x['label'])
df['sentiment_score'] = df['sentiment_analysis'].apply(lambda x: x['score'])

print("Sentiment analysis complete. Here are the first 5 rows with new sentiment columns:")
display(df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.countplot(x='sentiment_label', data=df, hue='sentiment_label', palette='viridis', legend=False)
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()